# 第15周 · 作业：β-VAE 扫描 + 论文复现报告（预计约7小时）

两件事：

1. **研究实验**（约 4 小时）：在 FashionMNIST 上，KL 项的权重 β 取 {0.1, 1, 4}，各跑 3 个种子，比较**重建误差**与**潜空间结构**（用潜空间上的 kNN 分类准确率衡量）。β-VAE 的出处：Higgins et al., *β-VAE: Learning Basic Visual Concepts with a Constrained Variational Framework*, ICLR 2017——β = 1 就是标准 VAE，β > 1 收紧潜变量的"信息通道"。
2. **复现报告**（约 3 小时）：按 `templates/replication_report.md` 写 2 页报告，填在本 notebook 最后（或单独写一个同样结构的 markdown 文件）。

| 部分 | 内容 | 建议时间 |
| --- | --- | --- |
| A | 实验设计 + 预先写下的假设 | 0.5 小时 |
| B | 执行：3 个 β × 3 个种子（CPU 上整套不到 1 分钟） | 1.5 小时 |
| C | 结果表、误差棒图、潜空间图 + 分析问题 | 1 小时 |
| D | 实验记录 | 0.5 小时 |
| E | 复现报告（2 页） | 3 小时 |
| F | 代码讲解 | 0.5 小时 |

**约定**：可以使用 AI 编程助手，但一定要写 F 部分的"代码讲解"。**自评**：研究实验用研究实验自评表（设计 30% · 执行 30% · 分析 30% · 表达 10%）；复现报告用复现报告自评表（都在学习指南里）（复现范围 20% · 结果对比 40% · 差异分析 30% · 可复现性 10%）。

In [ ]:
# 运行这一格来安装课程工具包（每次在 Colab 打开 notebook 后运行一次）
import importlib.util, subprocess, sys
if importlib.util.find_spec("zero2researcher") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ReaganGen/Zero2Researcher"], check=True)
print("课程工具包已就绪")

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from zero2researcher import data, experiments
from zero2researcher.checks import week15 as ck

device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available() else "cpu")
print("torch", torch.__version__, "| device:", device)

## A · 实验设计（先写，再跑）

实验规格：

- **科学变量**：β ∈ {0.1, 1, 4}，训练损失 = BCE + β·KL（每个 batch 求和）；
- **固定**：pytorch/examples 的 VAE 结构（784→400→20→400→784）、潜维度 20、Adam 1e-3、batch 128、**3 个 epoch**；训练集 = FashionMNIST 训练集前 1 万张；测试集 = 全部 1 万张；
- **种子** 0、1、2；
- **指标**（都在测试集上）：
  - 重建误差：每张图的 BCE；
  - KL：每张图的 $\mathrm{KL}(q(z|x)\,\|\,\mathcal{N}(0, I))$——不论训练时 β 是多少，**评估时都按同一个公式算**，三个 β 才可比；
  - 潜空间结构：编码器输出的 $\mu$ 作为 20 维特征，`KNeighborsClassifier(n_neighbors=5)` 在训练集 1 万张的 $\mu$ 上拟合、在测试集的 $\mu$ 上算准确率（用 $\mu$ 而不是采样的 $z$：评估要确定性）；
  - 附加：活跃维度个数（每维平均 KL > 0.1 的维度数，讲义 S3 的后验坍塌指标）。
- **对照**：同一份像素上的 PCA（20 个主成分）+ 同样的 kNN，作为"线性压缩到 20 维"的参照。

**A1 · 超参数分类**（科学 / 冗余 / 固定各至少写一个；本实验有没有需要为每个 β 单独调的冗余超参数？）

- 科学超参数：
- 冗余超参数：
- 固定超参数：

**A2 · 假设**（β 增大时，BCE、KL、kNN 准确率各自往哪个方向变？为什么？什么结果会推翻你的假设？）

-

## B · 执行实验

### B1 · 数据与模型（已给出）

In [ ]:
def load_dataset(cls, train):
    for attempt in (1, 2):
        try:
            return cls(root=data.download_root(), train=train, download=True)
        except Exception as e:
            if attempt == 1:
                print("下载失败，重试一次：", e)
            else:
                raise


fm_train, fm_test = load_dataset(datasets.FashionMNIST, True), load_dataset(datasets.FashionMNIST, False)
X_train = fm_train.data[:10000].float().div(255).reshape(-1, 784)
y_train = fm_train.targets[:10000]
X_test = fm_test.data.float().div(255).reshape(-1, 784)
y_test = fm_test.targets
print("训练", tuple(X_train.shape), "| 测试", tuple(X_test.shape), "| 类别", fm_train.classes)

BETAS, SEEDS, EPOCHS, Z_DIM = (0.1, 1, 4), (0, 1, 2), 3, 20


class VAE(nn.Module):
    # pytorch/examples/vae 的结构（BSD-3-Clause）
    def __init__(self, z_dim=Z_DIM):
        super().__init__()
        self.fc1, self.fc21, self.fc22 = nn.Linear(784, 400), nn.Linear(400, z_dim), nn.Linear(400, z_dim)
        self.fc3, self.fc4 = nn.Linear(z_dim, 400), nn.Linear(400, 784)

    def encode(self, x):
        h = F.relu(self.fc1(x))
        return self.fc21(h), self.fc22(h)

    def decode(self, z):
        return torch.sigmoid(self.fc4(F.relu(self.fc3(z))))

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)
        return self.decode(z), mu, logvar


def kl_per_dim(mu, logvar):
    # (batch, d) → 每个元素的 KL，与 -0.5 * (1 + logvar - mu² - exp(logvar)) 相同
    return -0.5 * (1 + logvar - mu.pow(2) - logvar.exp())

### B2 · 训练与评估一个配置

写 `run_config(beta, seed)`：固定种子 → 建 `VAE` → 用损失 `BCE(sum) + beta * KL(sum)` 训练 `EPOCHS` 个 epoch（Adam 1e-3、batch 128）→ 在测试集上返回一个 dict：

```python
{"bce": 每张图的 BCE, "kl": 每张图的 KL, "knn": μ 上的 5-近邻测试准确率, "active": KL/维 > 0.1 的维度个数,
 "model": 训练好的模型}
```

评估时 BCE 用一次采样的 $z$ 计算（和讲义、pytorch/examples 一致）；kNN 与活跃维度用 $\mu$。`"model"` 留给 C 部分画潜空间图。

In [ ]:
def encode_mu(model, X):
    model.eval()
    with torch.no_grad():
        return model.encode(X.to(device))[0].cpu().numpy()


def run_config(beta, seed):
    # 在这里写你的代码
    raise NotImplementedError


t0 = time.time()
smoke = run_config(1, 0)
print(f"冒烟测试 β=1、种子 0：BCE {smoke['bce']:.1f} | KL {smoke['kl']:.1f} | kNN {smoke['knn']:.4f} | "
      f"活跃维度 {smoke['active']}（{time.time() - t0:.1f} 秒）")
ck.knn_latent_accuracy(smoke["knn"])

### B3 · 主循环

对每个 β、每个种子调用 `run_config`，存成下面的结构（自查会检查结构与趋势）：

```python
results[beta] = {"bce": [3 个种子], "kl": [...], "knn": [...], "active": [...]}
```

顺便把每个 β 的种子 0 模型存进 `models[beta]`，C 部分画图要用。

In [ ]:
results, models = {}, {}
t0 = time.time()
# 在这里写你的代码
raise NotImplementedError
print(f"全部 {len(BETAS) * len(SEEDS)} 次训练用时 {time.time() - t0:.1f} 秒")
for beta, r in results.items():
    print(f"β = {beta:<3}: BCE {experiments.format_result(r['bce'], 1)} | KL {experiments.format_result(r['kl'], 2)} | "
          f"kNN {experiments.format_result(r['knn'])} | 活跃维度 {r['active']}")
ck.beta_trend(results)

### B4 · PCA 对照

在同一份训练像素上拟合 20 个主成分，用同样的 5-近邻在测试集上算准确率（确定性的，一次就够）。

In [ ]:
# 在这里写你的代码
raise NotImplementedError
print(f"PCA（20 维）+ 5-近邻：测试准确率 {pca_knn:.4f}")

## C · 结果表与图

**C1** · 结果表 `table`：行 = β（外加一行 "PCA-20"），列 = BCE/张、KL/张、kNN 准确率（均值 ± 标准差）、活跃维度（3 个种子）。

In [ ]:
# 在这里写你的代码
raise NotImplementedError
table

**C2** · 三栏误差棒图：横轴 β（对数轴），纵轴分别是 BCE/张、KL/张、kNN 准确率（误差棒 = 标准差）；第三栏把 PCA-20 画成水平虚线。总标题写结论。

In [ ]:
# 在这里写你的代码
raise NotImplementedError


**C3** · 潜空间图：对 β = 0.1 和 β = 4 的种子 0 模型，取测试集前 3000 张的 $\mu$（20 维），用 PCA 降到 2 维后按类别着色，两图并排。再各画一行重建图（同样 10 张测试图）。

In [ ]:
# 在这里写你的代码
raise NotImplementedError


**C4 · 分析问题**（每题两三句话，引用你表里的数字）

1. BCE 与 KL 随 β 的变化方向是什么？β 从 0.1 到 4，两者各变了多少？差异相对于种子间标准差大不大？
2. kNN 准确率随 β 怎样变化？β = 0.1 的 VAE 与 PCA-20 相比如何？"潜空间结构更好"在这里该如何定义——kNN 准确率衡量的是什么，又**没有**衡量什么（提示：β-VAE 论文关心的是"解耦"）？
3. 活跃维度随 β 怎样变化？把它和讲义 S3 的"后验坍塌"联系起来。
4. 本实验只训练了 3 个 epoch。你认为更长的训练会改变哪个结论、不会改变哪个？设计一个只改一个变量的后续实验来检验。
5. 评估时 BCE 用的是一次采样的 $z$，kNN 用的是 $\mu$。如果 BCE 也改用 $\mu$ 解码，β 的效应会变大还是变小？为什么？

（在这里回答）

## D · 实验记录（正式格式）

"结果"一栏必须是 3 个种子的均值 ± 标准差，并引用 C 部分的表和图；"局限"一栏讨论固定超参数（3 epochs、潜维度 20、1 万张训练子集、学习率未按 β 调）对结论的限定。

## 实验记录 #4 · 日期：____

- 研究问题：
- 假设（预期结果，以及什么结果会推翻它）：
- 自变量（这次只改变什么）：
- 控制变量（哪些保持不变；哪些干扰变量单独调优）：
- 基线：
- 数据与划分方式（如何避免泄漏）：
- 随机种子：
- 结果（数值：均值±标准差；图：链接或插图）：
- 解释：
- 局限：
- 下一步：

## E · 复现报告（2 页）

按 `templates/replication_report.md` 填写下面的骨架（每节的详细提示与自查清单见模板文件）。图请插入你复现 notebook 导出的 PNG，或给出链接；表格里的每个数字都要能在你的复现 notebook 输出里找到。

# 论文复现报告

- 姓名：
- 论文（作者，年份，标题，会议/期刊，arXiv 号或 DOI）：
- 复现目标（复现了原文的哪张图 / 哪个表的哪几行）：
- notebook 链接：

## 1 · 复现范围

**原文设置 vs 我的设置**

| 项目 | 原文设置（出处） | 我的设置 | 相同 / 不同 |
| --- | --- | --- | --- |
| 数据集与划分 | | | |
| 预处理 | | | |
| 模型结构 | | | |
| 优化器、学习率与调度 | | | |
| batch size / 训练步数 | | | |
| 评估指标与口径 | | | |
| 种子个数 | | | |

**有意的简化**（及预计影响的方向）：

**复现口径**：数值 / 趋势（理由）：

## 2 · 结果对比

| 设置 / 方法 | 原文数值 | 我的结果（均值 ± 标准差，n） | 绝对差 | 相对差 | 差异 > 1 个标准差？ |
| --- | --- | --- | --- | --- | --- |
| | | | | | |

（对比图）

**结论**：

## 3 · 差异分析

| # | 观察到的差异 | 假设的原因 | 检验方法（只改一个变量） | 假设成立时的预期 | 已检验？结果 |
| --- | --- | --- | --- | --- | --- |
| 1 | | | | | |
| 2 | | | | | |

## 4 · 可复现性

- [ ] notebook 一键运行（日期、耗时、硬件）：
- [ ] 依赖版本与随机种子：
- [ ] 数据自动下载 / 加载：
- [ ] 报告中的每个数字都能在 notebook 输出里找到

## 5 · 参考文献

1.

## F · 代码讲解（每份作业都写）

用你自己的话写一段（约 150–300 字）：

1. 逐段解释 `run_config`：训练损失里 β 乘在哪一项上；评估时 KL 为什么**不**乘 β；kNN 为什么用 $\mu$、为什么在训练集上拟合、测试集上评估；
2. 注明哪些代码由 AI 助手生成、你对它做了哪些检查或修改；
3. 写出一处你一开始没看懂、后来弄懂的地方。

（在这里写）